<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/final_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd
import subprocess
import tempfile
import shutil
import torch
import json
import time
import os

if os.name == "nt":
    userdata = None
else:
    from google.colab import userdata

# Clone Repository

In [ ]:
IS_LOCAL = os.name == "nt"

LOCAL_REPO_PATH = Path.home() / "Corpus2GeoRAG"
repo_dir = (
    LOCAL_REPO_PATH
    if IS_LOCAL
    else Path("/content/Corpus2GeoRAG")
)

REPO_PATH = str(repo_dir)

if not repo_dir.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/EtzionR/Corpus2GeoRAG.git",
            REPO_PATH
        ],
        check=True
    )
elif not (repo_dir / ".git").exists():
    raise RuntimeError(f"{REPO_PATH} exists but is not a Git checkout.")

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Working folder:", REPO_PATH)

# Data Upload

In [ ]:
# Locate the file inside the repository.
input_path = Path("ApplyNER/data_initial_cleaning.json")

# Load the complete dataset.
with input_path.open("r", encoding="utf-8") as f:
    data_processed = json.load(f)

# Verify the loaded data.
print(f"Loaded {len(data_processed)} pages.")
print(f"Total paragraphs: {sum(len(p) for p in data_processed.values())}")
print(f"Source: {input_path.resolve()}")

# model loading

chosen model: DistillBERT
date: 12.10.26

In [ ]:
MODEL_DistilBERT = "dslim/distilbert-NER"

ner_DistilBERT = pipeline(
    "ner",
    model=MODEL_DistilBERT,
    aggregation_strategy="average",
    device=0 if torch.cuda.is_available() else -1
)

print("Model device:", ner_DistilBERT.device)

ner_DistilBERT

# Model Application

In [ ]:
BATCH_SIZE = 32 if ner_DistilBERT.device.type == "cuda" else 1

# Use the full dataset.
paragraph_records = [
    (page, paragraph_id, paragraph)
    for page, paragraphs in data_processed.items()
    for paragraph_id, paragraph in enumerate(paragraphs)
]

rows = []
started = time.perf_counter()

predictions = ner_DistilBERT(
    (record[2] for record in paragraph_records),
    batch_size=BATCH_SIZE,
    num_workers=0
)

for (page, paragraph_id, paragraph), entities in tqdm(
    zip(paragraph_records, predictions, strict=True),
    total=len(paragraph_records),
    desc="DistilBERT NER — full data",
    unit="paragraph"
):
    for entity in entities:
        rows.append({
            "entity_group": entity["entity_group"],
            "score": entity["score"],
            "word": entity["word"],
            "start": entity["start"],
            "end": entity["end"],
            "Source": page,
            "Index": paragraph_id,
            "Text": paragraph
        })

elapsed = time.perf_counter() - started

columns = [
    "entity_group",
    "score",
    "word",
    "start",
    "end",
    "Source",
    "Index",
    "Text"
]

df_DistilBERT = pd.DataFrame(rows, columns=columns)

output_dir = "ApplyNER/NER_output"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "post_NER_final.parquet")
df_DistilBERT.to_parquet(
    output_path,
    engine="pyarrow",
    compression="zstd",
    index=False
)

print(
    f"Processed {len(data_processed)} pages "
    f"and {len(paragraph_records)} paragraphs in {elapsed:.1f} seconds."
)
print(f"Model device: {ner_DistilBERT.device}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Saved {len(df_DistilBERT)} entity rows to {output_path}")

# Git Push/Commit

In [ ]:
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

LOCAL_REPO_PATH = Path.home() / GITHUB_REPO
MAX_GIT_FILE_BYTES = 100 * 1024 * 1024

# --------------------------------------------------
# Detect runtime
# --------------------------------------------------

if os.name == "nt":
    IS_LOCAL = True
elif os.environ.get("COLAB_RELEASE_TAG"):
    IS_LOCAL = False
else:
    raise RuntimeError(
        "Could not identify this runtime as Windows or a hosted Colab VM."
    )

if shutil.which("git") is None:
    raise RuntimeError(
        "Git is not available to this Jupyter environment. "
        "Install Git for Windows and restart Jupyter if running locally."
    )

# --------------------------------------------------
# Locate the repository
# --------------------------------------------------

if IS_LOCAL:
    candidates = [Path.cwd(), LOCAL_REPO_PATH]

    previous_repo_path = globals().get("REPO_PATH")
    if previous_repo_path:
        candidates.insert(0, Path(previous_repo_path))

    detected_repo = None

    for candidate in candidates:
        if not candidate.is_dir():
            continue

        probe = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=candidate,
            text=True,
            encoding="utf-8",
            errors="replace",
            capture_output=True
        )

        if probe.returncode == 0:
            root = Path(probe.stdout.strip()).resolve()

            if root.name.lower() == GITHUB_REPO.lower():
                detected_repo = root
                break

    if detected_repo is None:
        raise FileNotFoundError(
            f"Could not find your local {GITHUB_REPO} checkout.\n"
            f"Clone it to {LOCAL_REPO_PATH}, or update LOCAL_REPO_PATH."
        )

    REPO_PATH = str(detected_repo)

else:
    REPO_PATH = f"/content/{GITHUB_REPO}"

    if not Path(REPO_PATH).is_dir():
        raise FileNotFoundError(
            f"{REPO_PATH} does not exist. Run your clone cell first."
        )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Repository:", REPO_PATH)


def run_git(args, *, env=None, show_output=True):
    """Run Git and include its actual output if it fails."""
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_PATH,
        env=env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if show_output:
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, end="")

    if result.returncode != 0:
        details = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"git {' '.join(args)} failed:\n{details}")

    return result


# --------------------------------------------------
# Git configuration
# --------------------------------------------------

run_git(["config", "user.name", GITHUB_USER])
run_git(["config", "user.email", GITHUB_EMAIL])
run_git([
    "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
])

branch = run_git(
    ["branch", "--show-current"],
    show_output=False
).stdout.strip()

if not branch:
    raise RuntimeError("Could not determine the current Git branch.")

print("Branch:", branch)

# Stop if an earlier rebase or merge is still paused.
for state_path in ("rebase-merge", "rebase-apply", "MERGE_HEAD"):
    git_state = run_git(
        ["rev-parse", "--git-path", state_path],
        show_output=False
    ).stdout.strip()

    if Path(git_state).exists():
        raise RuntimeError(
            "An earlier rebase or merge is still in progress. "
            "Check `git status` and resolve it before rerunning this cell."
        )

# --------------------------------------------------
# Authentication and synchronization
# --------------------------------------------------

git_env = os.environ.copy()
git_env["GIT_TERMINAL_PROMPT"] = "0"

auth_options = []
askpass_dir = None

try:
    if IS_LOCAL:
        # Use Git credentials already configured on your computer.
        print("Authentication: local Git credentials.")

    else:
        from google.colab import userdata

        github_token = userdata.get("GITHUB_TOKEN")

        if not github_token:
            raise ValueError(
                "GITHUB_TOKEN is missing from Colab Secrets. "
                "Add it and enable notebook access."
            )

        askpass_dir = tempfile.TemporaryDirectory(
            prefix="github-askpass-"
        )
        askpass_path = Path(askpass_dir.name) / "askpass.sh"

        askpass_path.write_text(
            "#!/bin/sh\n"
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
            '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            "esac\n",
            encoding="utf-8"
        )
        askpass_path.chmod(0o700)

        git_env["GITHUB_USER"] = GITHUB_USER
        git_env["GITHUB_TOKEN"] = github_token
        git_env["GIT_ASKPASS"] = str(askpass_path)

        auth_options = ["-c", "credential.helper="]

        print("Authentication: Colab GITHUB_TOKEN Secret.")

    # --------------------------------------------------
    # Prevent accidental nested repository
    # --------------------------------------------------

    ignore_entry = f"/{GITHUB_REPO}/"
    gitignore_path = Path(REPO_PATH) / ".gitignore"

    existing_ignore = (
        gitignore_path.read_text(encoding="utf-8")
        if gitignore_path.exists()
        else ""
    )

    if ignore_entry not in existing_ignore.splitlines():
        with gitignore_path.open("a", encoding="utf-8") as f:
            if existing_ignore and not existing_ignore.endswith("\n"):
                f.write("\n")
            f.write(ignore_entry + "\n")

    nested_repo = run_git(
        ["ls-files", "--stage", "--", GITHUB_REPO],
        show_output=False
    ).stdout.strip()

    if nested_repo:
        run_git([
            "rm", "--cached", "-r", "-f", "--", GITHUB_REPO
        ])
        print("Removed the nested repository path from Git tracking.")

    # --------------------------------------------------
    # Find files exceeding GitHub's per-file limit
    # --------------------------------------------------

    candidate_files = run_git(
        [
            "ls-files",
            "--cached",
            "--others",
            "--exclude-standard",
            "-z"
        ],
        show_output=False
    ).stdout.split("\0")

    excluded_files = set()

    for relative_path in sorted(set(candidate_files) - {""}):
        file_path = Path(REPO_PATH) / relative_path

        if not file_path.is_file():
            continue

        file_size = file_path.stat().st_size

        if file_size > MAX_GIT_FILE_BYTES:
            excluded_files.add(relative_path)

            # Remove from Git's index, while keeping the file on disk.
            run_git(
                [
                    "rm",
                    "--cached",
                    "--ignore-unmatch",
                    "-f",
                    "--",
                    f":(top,literal){relative_path}"
                ],
                show_output=False
            )

            print(
                f"Excluded oversized file: {relative_path} "
                f"({file_size / 1024**2:.2f} MiB). "
                "Kept on disk."
            )

    # --------------------------------------------------
    # Stage changes, excluding oversized files
    # --------------------------------------------------

    exclusion_pathspecs = [
        f":(top,literal,exclude){path}"
        for path in sorted(excluded_files)
    ]

    run_git(["add", "-A", "--", ".", *exclusion_pathspecs])

    # Exclude notebook copies, preserving the original cell's behavior.
    staged_files = run_git(
        ["diff", "--cached", "--name-only", "-z"],
        show_output=False
    ).stdout.split("\0")

    for notebook in staged_files:
        if (
            notebook.endswith(".ipynb")
            and notebook not in excluded_files
        ):
            run_git([
                "restore", "--staged", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Create a normal commit
    # --------------------------------------------------

    staged_diff = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_PATH,
        stdin=subprocess.DEVNULL
    )

    if staged_diff.returncode == 1:
        run_git(["commit", "-m", COMMIT_MESSAGE])
        print("Repository changes committed.")
    elif staged_diff.returncode == 0:
        print("No new repository changes to commit.")
    else:
        raise RuntimeError("Could not check the staged changes.")

    # --------------------------------------------------
    # Restore tracked notebook copies on disk
    # This discards edits to those on-disk copies.
    # --------------------------------------------------

    tracked_files = run_git(
        ["ls-tree", "-r", "--name-only", "-z", "HEAD"],
        show_output=False
    ).stdout.split("\0")

    for notebook in tracked_files:
        if notebook.endswith(".ipynb"):
            run_git([
                "restore", "--worktree", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Pull/rebase GitHub changes
    # --------------------------------------------------

    pull = subprocess.run(
        ["git", *auth_options, "pull", "--rebase", "origin", branch],
        cwd=REPO_PATH,
        env=git_env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if pull.stdout:
        print(pull.stdout, end="")
    if pull.stderr:
        print(pull.stderr, end="")

    if pull.returncode != 0:
        conflicted = run_git(
            ["diff", "--name-only", "--diff-filter=U"],
            show_output=False
        ).stdout.splitlines()

        if conflicted == [".gitignore"]:
            ours = run_git(
                ["show", ":2:.gitignore"],
                show_output=False
            ).stdout

            theirs = run_git(
                ["show", ":3:.gitignore"],
                show_output=False
            ).stdout

            if ours.strip() != theirs.strip():
                raise RuntimeError(
                    "The .gitignore versions differ. The rebase is paused; "
                    "inspect the conflict before continuing."
                )

            gitignore_path.write_text(
                ours + ("\n" if ours and not ours.endswith("\n") else ""),
                encoding="utf-8"
            )

            run_git(["add", "--", ".gitignore"])
            run_git([
                "-c", "core.editor=true", "rebase", "--continue"
            ])

        else:
            raise RuntimeError(
                "Pull failed. Check the Git output above and `git status`. "
                "If running locally, verify your GitHub authentication."
            )

    # --------------------------------------------------
    # Push synchronized changes
    # --------------------------------------------------

    run_git(
        [*auth_options, "push", "origin", branch],
        env=git_env
    )

    print(
        f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} "
        f"on branch '{branch}'."
    )

finally:
    if askpass_dir is not None:
        askpass_dir.cleanup()

    git_env.pop("GITHUB_TOKEN", None)